In [2]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain.chains import RetrievalQA

llm = ChatOpenAI()

cache_dir = LocalFileStore("./.cache/")


splitter = CharacterTextSplitter.from_tiktoken_encoder(
  separator='\n',
  chunk_size=600,
  chunk_overlap=100,
)

loader = UnstructuredFileLoader("./files/chapter_one.docx")

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
  embeddings, cache_dir
)


vectorstore = FAISS.from_documents(docs, cached_embeddings)


chain = RetrievalQA.from_chain_type(
  llm=llm,
  chain_type="refine",
  retriever=vectorstore.as_retriever(),
)

chain.run("Where does Winston live?")

'Winston lives in a flat in Victory Mansions, a grim and oppressive apartment building in Airstrip One, a province of the totalitarian state of Oceania. The building is constantly monitored by telescreens that broadcast propaganda and spy on the residents, enforcing the strict rules and surveillance of the Party. The Ministry of Love, a terrifying and heavily fortified government building, is nearby, making it a place impossible to enter except on official business. Winston must constantly maintain a facade of quiet optimism when facing the telescreen, even though he engages in rebellious activities like reading forbidden books. Despite the bleak conditions and lack of food, Winston finds some solace in drinking Victory Gin and smoking Victory Cigarettes as he secretly records his thoughts and experiences in a hidden journal.'